In [2]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import logging
from sklearn.preprocessing import MinMaxScaler
import spectral
import random
from skimage import color
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
import copy
import os

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# Set a fixed seed for reproducibility
seed = 10
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)  # If using GPU

# Ensure deterministic behavior
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

###########################################
# Helper functions
###########################################
def process_rgb(cube, bands, ill, CMFs):
    """
    Converts hyperspectral cube data to RGB using XYZ conversion.
    """
    ill_interp = np.interp(bands, ill[:, 0], ill[:, 1])
    CMFs_interp = np.column_stack([
        np.interp(bands, CMFs[:, 0], CMFs[:, 1]),
        np.interp(bands, CMFs[:, 0], CMFs[:, 2]),
        np.interp(bands, CMFs[:, 0], CMFs[:, 3])
    ])
    sp_tristREF = CMFs_interp * ill_interp[:, None]
    xyz = np.dot(cube, sp_tristREF) / np.sum(sp_tristREF[:, 1], axis=0)
    
    # Convert XYZ to RGB
    rgb = color.xyz2rgb(xyz)
    return rgb

def srgb_to_linear(x):
    # x is assumed to be in [0,1]
    threshold = 0.04045
    return torch.where(x <= threshold, x / 12.92, ((x + 0.055) / 1.055) ** 2.4)

def rgb_to_xyz_torch(rgb):
    """
    Convert sRGB (in [0,1]) to XYZ.
    Assumes rgb is a tensor of shape (N,3) with values in [0,1].
    """
    # First convert to linear RGB
    rgb_lin = srgb_to_linear(rgb)
    # sRGB to XYZ matrix (D65)
    M = torch.tensor([[0.4124564, 0.3575761, 0.1804375],
                      [0.2126729, 0.7151522, 0.0721750],
                      [0.0193339, 0.1191920, 0.9503041]], dtype=rgb.dtype, device=rgb.device)
    xyz = torch.matmul(rgb_lin, M.t())
    return xyz

def f_xyz(t):
    delta = 6/29
    delta3 = delta**3
    return torch.where(t > delta3, t ** (1/3), t / (3 * delta**2) + 4/29)

def xyz_to_lab_torch(xyz):
    # D65 white point
    Xn, Yn, Zn = 0.95047, 1.00000, 1.08883
    X = xyz[:, 0] / Xn
    Y = xyz[:, 1] / Yn
    Z = xyz[:, 2] / Zn
    fX = f_xyz(X)
    fY = f_xyz(Y)
    fZ = f_xyz(Z)
    L = 116 * fY - 16
    a = 500 * (fX - fY)
    b = 200 * (fY - fZ)
    lab = torch.stack([L, a, b], dim=1)
    return lab

def rgb_to_lab_torch(rgb):
    """
    Converts a tensor of RGB values (shape (N,3) in [0,1]) to Lab.
    """
    xyz = rgb_to_xyz_torch(rgb)
    lab = xyz_to_lab_torch(xyz)
    return lab


# Directory containing all .hdr files
input_directory = '../../data/colorChecker_SG/cubes/'
output_directory = '../results/plots_save_state/'

# Create the output directory if it doesn't exist
os.makedirs(output_directory, exist_ok=True)

# Load the reference data
ill = np.loadtxt('../../data/CIE_D65.txt')          
CMFs = np.loadtxt('../../data/CIE2degCMFs_1931.txt')
cube_ref = spectral.open_image('../../data/colorChecker_SG/cubeCC_DigitalSG_REF.hdr')
cube_ref_data = cube_ref.load()
wl_ref = np.array(cube_ref.metadata['wavelength'], dtype=float)

class SimpleMLP(nn.Module):
    def __init__(self, input_size=3, hidden_size=128, output_size=3):
        super(SimpleMLP, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, output_size)
    def forward(self, x):
        return self.fc2(self.relu(self.fc1(x)))
    
    
    

class HybridRGBLoss(nn.Module):
    def __init__(self, alpha=0.5, eps=1e-6):
        super(HybridRGBLoss, self).__init__()
        self.alpha = alpha  
        self.eps = eps  
        self.mse_loss = nn.MSELoss()

    def forward(self, y_pred, y_true):
        y_pred_norm = torch.nn.functional.normalize(y_pred, p=2, dim=1, eps=self.eps)
        y_true_norm = torch.nn.functional.normalize(y_true, p=2, dim=1, eps=self.eps)
        cosine_similarity = torch.sum(y_pred_norm * y_true_norm, dim=1)
        cosine_similarity = torch.clamp(cosine_similarity, -1.0, 1.0)
        angular_loss = 1 - cosine_similarity  
        mse_loss = self.mse_loss(y_pred, y_true)
        total_loss = self.alpha * angular_loss.mean() + (1 - self.alpha) * mse_loss
        return total_loss
    
class HybridMSEDeltaELoss(nn.Module):
    def __init__(self, alpha=0.5, eps=1e-6, scaler_mean=None, scaler_scale=None):
        """
        alpha: weight for MSE (0 <= alpha <= 1). The delta E loss weight is (1 - alpha).
        scaler_mean, scaler_scale: numpy arrays (shape (3,)) from scaler_ref for un-standardizing.
        """
        super(HybridMSEDeltaELoss, self).__init__()
        self.alpha = alpha
        self.eps = eps
        self.mse_loss = nn.MSELoss()
        # Convert scaler parameters to torch tensors
        if scaler_mean is None or scaler_scale is None:
            raise ValueError("scaler_mean and scaler_scale must be provided")
        self.register_buffer('mean', torch.tensor(scaler_mean, dtype=torch.float32))
        self.register_buffer('scale', torch.tensor(scaler_scale, dtype=torch.float32))
    
    def forward(self, y_pred, y_true):
        # Un-standardize (linear transformation) so that:
        # original_rgb = standardized * scale + mean
        # Here both y_pred and y_true are of shape (N, 3)
        y_pred_orig = y_pred * self.scale + self.mean
        y_true_orig = y_true * self.scale + self.mean
        # Clamp values to [0,1] (if needed)
        y_pred_orig = torch.clamp(y_pred_orig, 0.0, 1.0)
        y_true_orig = torch.clamp(y_true_orig, 0.0, 1.0)
        
        # Compute MSE on original RGB values
        mse = self.mse_loss(y_pred_orig, y_true_orig)
        
        # Convert RGB to Lab (each of shape (N,3))
        lab_pred = rgb_to_lab_torch(y_pred_orig)
        lab_true = rgb_to_lab_torch(y_true_orig)
        # Compute delta E as Euclidean distance in Lab space
        delta_e = torch.norm(lab_pred - lab_true, p=2, dim=1)  # shape (N,)
        delta_e_loss = torch.mean(delta_e)
        
        # Total hybrid loss
        total_loss = self.alpha * mse + (1 - self.alpha) * delta_e_loss
        return total_loss

class EarlyStopping:
    def __init__(self, patience=10, delta=0, verbose=False, path='best_model.pth'):
        self.patience = patience  # Number of epochs to wait for improvement
        self.delta = delta  # Minimum change to qualify as an improvement
        self.verbose = verbose  # Whether to print the stop message
        self.path = path  # Path to save the best model
        self.best_loss = None  # Best validation loss
        self.counter = 0  # Counter for how many epochs without improvement
        self.early_stop = False  # Flag to stop the training
        self.best_model_wts = None  # To store the best model's weights

    def __call__(self, val_loss, model):
        if self.best_loss is None:
            self.best_loss = val_loss
            self.best_model_wts = copy.deepcopy(model.state_dict())
        elif val_loss < self.best_loss - self.delta:
            self.best_loss = val_loss
            self.best_model_wts = copy.deepcopy(model.state_dict())
            self.counter = 0  # Reset the counter
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
                # if self.verbose:
                #     print(f"Early stopping triggered after {self.counter} epochs with no improvement.")
                    
        return self.early_stop


# Process each .hdr file in the input directory
for file_name in os.listdir(input_directory):
    if file_name.endswith('.hdr'):
        # Full path to the cube
        cube_path = os.path.join(input_directory, file_name)
        
        # Load cube data
        logging.info(f'Processing file: {file_name}')
        cube = spectral.open_image(cube_path)
        cube_data = cube.load()
        wl_input = np.array(cube.metadata['wavelength'], dtype=float)

        # Process RGB data
        rgb_input = process_rgb(cube_data, wl_input, ill, CMFs)
        rgb_ref = process_rgb(cube_ref_data, wl_ref, ill, CMFs)

        # Normalize data in RGB space
        rgb_input_2d = rgb_input.reshape(-1, rgb_input.shape[-1])
        rgb_ref_2d = rgb_ref.reshape(-1, rgb_ref.shape[-1])

        scaler_input = StandardScaler()
        scaler_ref = StandardScaler()
        X_norm = scaler_input.fit_transform(rgb_input_2d)
        Y_norm = scaler_ref.fit_transform(rgb_ref_2d)

        X_full = X_norm.reshape(rgb_input.shape)
        Y_full = Y_norm.reshape(rgb_ref.shape)

        np.random.seed(seed)
        # Prepare training data
        X_flat = X_full.reshape(-1, 3)
        Y_flat = Y_full.reshape(-1, 3)

        n_pixels = X_flat.shape[0]
        train_size = int(0.8 * n_pixels)
        train_indices = np.random.choice(n_pixels, train_size, replace=False)
        test_indices = np.setdiff1d(np.arange(n_pixels), train_indices)

        X_train_split = X_flat[train_indices]
        X_test_split  = X_flat[test_indices]
        Y_train_split = Y_flat[train_indices]
        Y_test_split  = Y_flat[test_indices]

        X_train_torch = torch.tensor(X_train_split, dtype=torch.float32)
        Y_train_torch = torch.tensor(Y_train_split, dtype=torch.float32)
        X_test_torch  = torch.tensor(X_test_split, dtype=torch.float32)
        Y_test_torch  = torch.tensor(Y_test_split, dtype=torch.float32)


        # Define model and loss
        model = SimpleMLP(input_size=3, hidden_size=128, output_size=3)
        optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)
        # loss_function = nn.MSELoss()
        loss_function = HybridMSEDeltaELoss(alpha=1, scaler_mean=scaler_ref.mean_, scaler_scale=scaler_ref.scale_)
        scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=300, gamma=0.5)  # Reduce LR every 200 epochs
        early_stopping = EarlyStopping(patience=50, delta=0, verbose=True)

        # Training loop
        epochs = 2000
        batch_size = 4
        train_losses = []
        val_losses = []

        for epoch in range(epochs):
            model.train()
            perm = torch.randperm(X_train_torch.size(0))
            X_train_shuffled = X_train_torch[perm]
            Y_train_shuffled = Y_train_torch[perm]
            epoch_train_loss = 0.0
            
            for i in range(0, X_train_shuffled.size(0), batch_size):
                X_batch = X_train_shuffled[i:i+batch_size]
                Y_batch = Y_train_shuffled[i:i+batch_size]
                
                optimizer.zero_grad()
                Y_pred = model(X_batch)
                loss = loss_function(Y_pred, Y_batch)
                loss.backward()
                optimizer.step()

                epoch_train_loss += loss.item()
            
            scheduler.step()

            avg_train_loss = epoch_train_loss / (X_train_shuffled.size(0) // batch_size)
            train_losses.append(avg_train_loss)
            
            scheduler.step()

            model.eval()
            with torch.no_grad():
                Y_val_pred = model(X_test_torch)
                val_loss = loss_function(Y_val_pred, Y_test_torch).item()
            val_losses.append(val_loss)

            early_stopping(val_loss, model)
            # if early_stopping.early_stop:
            #     logging.info("Early stopping triggered.")
            #     break

        # Load the best model weights
        model.load_state_dict(early_stopping.best_model_wts)

        # Apply correction and evaluate with ΔE2000
        X_full_flat = X_full.reshape(-1, 3)
        corrected_flat = model(torch.tensor(X_full_flat, dtype=torch.float32)).detach().numpy()
        corrected_rgb = scaler_ref.inverse_transform(corrected_flat)
        corrected_rgb_image = corrected_rgb.reshape(rgb_ref.shape)

        corrected_lab = color.rgb2lab(corrected_rgb_image)
        lab_ref = color.rgb2lab(rgb_ref)

        error_map = color.deltaE_ciede2000(lab_ref, corrected_lab)

        # Compute mean & max ΔE2000 error for the test set
        error_map_flat = error_map.reshape(-1)
        test_error_values = error_map_flat[test_indices]
        mean_error_test = np.mean(test_error_values)
        max_error_test = np.max(test_error_values)
        logging.info(f'Mean ΔE2000 Error for {file_name}: {mean_error_test:.2f}')
        logging.info(f'Max ΔE2000 Error for {file_name}: {max_error_test:.2f}')

        # Get test positions in the image
        test_positions = np.unravel_index(test_indices, lab_ref.shape[:2])

        # Plot the ΔE2000 error map
        plt.figure(figsize=(8, 6))
        plt.imshow(error_map, cmap='jet', vmin=0, vmax=10)
        plt.colorbar(label='ΔE2000')
        plt.scatter(test_positions[1], test_positions[0], s=2, c='white', label='Test Patches', alpha=0.7)
        plt.title('ΔE2000 Error Map')

        # Add text annotations for Mean and Max Error and File name
        file_name_no_extension = os.path.splitext(file_name)[0]
        plt.text(0.5, 1.1, f'Cube: {file_name_no_extension}\nMean Error: {mean_error_test:.2f}\nMax Error: {max_error_test:.2f}', 
                 ha='center', va='bottom', transform=plt.gca().transAxes, fontsize=12, color='black')

        # Adjust layout and save the plot
        plt.tight_layout(pad=3.0)
        plot_filename = os.path.join(output_directory, f'{file_name_no_extension}_error_map.png')
        plt.savefig(plot_filename)
        plt.close()
        logging.info(f'Saved plot for {file_name} to {plot_filename}')


2025-04-01 13:03:53,172 - INFO - Processing file: cubeCC_120f-velvia-f8.hdr
2025-04-01 13:04:31,263 - INFO - Mean ΔE2000 Error for cubeCC_120f-velvia-f8.hdr: 3.52
2025-04-01 13:04:31,264 - INFO - Max ΔE2000 Error for cubeCC_120f-velvia-f8.hdr: 8.83
2025-04-01 13:04:31,362 - INFO - Saved plot for cubeCC_120f-velvia-f8.hdr to ../results/plots_save_state/cubeCC_120f-velvia-f8_error_map.png
2025-04-01 13:04:31,362 - INFO - Processing file: cubeCC_no6-ekta100-expo3.hdr
2025-04-01 13:05:09,315 - INFO - Mean ΔE2000 Error for cubeCC_no6-ekta100-expo3.hdr: 5.92
2025-04-01 13:05:09,315 - INFO - Max ΔE2000 Error for cubeCC_no6-ekta100-expo3.hdr: 16.81
2025-04-01 13:05:09,413 - INFO - Saved plot for cubeCC_no6-ekta100-expo3.hdr to ../results/plots_save_state/cubeCC_no6-ekta100-expo3_error_map.png
2025-04-01 13:05:09,413 - INFO - Processing file: cubeCC_fuji-frame13.hdr
2025-04-01 13:05:47,398 - INFO - Mean ΔE2000 Error for cubeCC_fuji-frame13.hdr: 3.10
2025-04-01 13:05:47,399 - INFO - Max ΔE2000 E